# Borehole lithology database from the BDR Word files

**Alwar groundwater study · 14-block domain (LGD 574–587)**

Quality control of the borehole database extracted from the Atal Bhujal (17) and NHP (42) BDR Word files: borehole identifiers, NHP coordinates (DMS in the Word files vs the Excel sheet), review of flagged lithology intervals against the original Word tables (source-preserving, no automatic boundary correction), and the borehole lithology elevation database (Steps 4–6).

> ℹ️ Some inputs were created outside this notebook and are not reproduced by any cell in the repository: `Alwar_BDR_Extracted_Database.xlsx`, `Borehole_Master_Combined.csv`, `Lithology_Intervals_Combined.csv`, `Alwar_Accepted_Lithology_Intervals.csv`, `Alwar_Lithology_Classified_Intervals_Final.csv`, `Step6B_Borehole_Coordinates_For_DEM.csv`, `Step6C_SRTM30m_Borehole_Elevations.csv`. See `docs/DATA_LINEAGE_GAPS.md`.

| | |
|---|---|
| **Reads** | `data/BDR/Alwar/Atalbdr/*.docx`, `data/BDR/Alwar/nhpbdr 2023/*.docx`, `Alwar_BDR_Extracted_Database.xlsx`, `Borehole_Master_Combined.csv`, `Lithology_Intervals_Combined.csv`, `Step6C_SRTM30m_Borehole_Elevations.csv` |
| **Writes** | `data/BDR/Alwar/…` (numbered audit files `06_…` to `12_…`, `Step6E_…`, `Step6F_Borehole_Lithology_Elevation_Database.csv`) |
| **Cells** | 20 code cells, copied unchanged from `alwargwd_data_analysis_2_2.ipynb` (outputs cleared) |

**How to run:** run the set-up cell, then each section from top to bottom. Every cell reads its inputs from Drive, except where a label says it needs the cell above.

Cell labels: `[orig #N]` = position in the original notebook · ✅ ran OK · ⚠ last run ended with an error · ⏸ interrupted · • output not saved.

**Contents**
- 1. Inventory of the BDR folders and the extracted database
- 2. NHP coordinates from the Word files; decision register
- 3. Lithology review against the original Word tables (Step 4)
- 4. Borehole lithology elevation database (Step 6)

In [ ]:
# ── SETUP — run this cell first ─────────────────────────────────────────────
import os, sys, json, subprocess
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

CFG = "/content/drive/MyDrive/gw_framework/repo_config.json"   # remembers your repository
if os.path.isfile(CFG):
    REPO = json.load(open(CFG))["repo"]
else:
    REPO = input("GitHub repository (username/repo-name): ").strip().strip("/")
    json.dump({"repo": REPO}, open(CFG, "w"))

REPO_DIR = "/content/" + REPO.split("/")[-1]
try:                                   # private repository: add a Colab secret named GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
elif subprocess.run(["git", "clone", "-q", URL, REPO_DIR]).returncode != 0:
    os.remove(CFG)
    raise RuntimeError(f"Could not clone '{REPO}'. Check the name (and GH_TOKEN if private), then run this cell again.")
sys.path.insert(0, REPO_DIR + "/src")

from alwar_gw import colab
PATHS = colab.setup(packages=["python-docx", "openpyxl"], modflow=False)
del TOKEN, URL


## 1. Inventory of the BDR folders and the extracted database

**[orig #255]** Inventory of `Alwar_BDR_Lithology_Inventory` (folders, file types, CSV and PDF summaries)  
<sub>• output not saved in the source notebook</sub>

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import pandas as pd
from pathlib import Path
from collections import defaultdict

BASE = Path(
    "/content/drive/MyDrive/Alwar_BDR_Lithology_Inventory"
)

assert BASE.exists(), f"Folder not found: {BASE}"

OUTPUT = BASE / "09_COMPLETE_FOLDER_INSPECTION"
OUTPUT.mkdir(exist_ok=True)

# --------------------------------------------------
# 1. RECURSIVE FOLDER AND FILE INVENTORY
# --------------------------------------------------

inventory = []
folder_summary = []

for root, dirs, files in os.walk(BASE):
    dirs[:] = sorted(
        d for d in dirs
        if d != "09_COMPLETE_FOLDER_INSPECTION"
    )

    root_path = Path(root)
    relative_folder = str(root_path.relative_to(BASE))

    folder_summary.append({
        "Folder": relative_folder,
        "Subfolder_Count": len(dirs),
        "File_Count": len(files)
    })

    for filename in sorted(files):
        path = root_path / filename

        inventory.append({
            "Folder": relative_folder,
            "Filename": filename,
            "Extension": path.suffix.lower(),
            "Size_MB": round(path.stat().st_size / (1024**2), 4),
            "Full_Path": str(path)
        })

inventory_df = pd.DataFrame(inventory)
folders_df = pd.DataFrame(folder_summary)

print("\nTOTAL FOLDERS:", len(folders_df))
print("TOTAL FILES:", len(inventory_df))

print("\n========== COMPLETE FOLDER TREE ==========")

for root, dirs, files in os.walk(BASE):
    dirs[:] = sorted(
        d for d in dirs
        if d != "09_COMPLETE_FOLDER_INSPECTION"
    )

    level = len(Path(root).relative_to(BASE).parts)
    print("  " * level + Path(root).name + "/")

    for f in sorted(files):
        print("  " * (level + 1) + f)

# --------------------------------------------------
# 2. FILE TYPE SUMMARY
# --------------------------------------------------

print("\n========== FILE TYPE SUMMARY ==========")

print(
    inventory_df.groupby("Extension")
    .agg(
        File_Count=("Filename", "count"),
        Total_Size_MB=("Size_MB", "sum")
    )
    .sort_values("File_Count", ascending=False)
)

# --------------------------------------------------
# 3. INSPECT CSV FILES
# --------------------------------------------------

csv_reports = []

for _, row in inventory_df.iterrows():

    path = Path(row["Full_Path"])

    if path.suffix.lower() != ".csv":
        continue

    print("\n" + "="*100)
    print("CSV:", path.relative_to(BASE))
    print("="*100)

    try:
        df = pd.read_csv(path, low_memory=False)

        print("Rows:", len(df))
        print("Columns:", len(df.columns))
        print("Column names:", df.columns.tolist())

        print("\nMissing values:")
        print(df.isna().sum().to_dict())

        print("\nSample records:")
        print(df.head(3).to_string(index=False))

        csv_reports.append({
            "File": str(path.relative_to(BASE)),
            "Rows": len(df),
            "Columns_Count": len(df.columns),
            "Columns": " | ".join(df.columns.astype(str)),
            "Missing_Values_Total": int(df.isna().sum().sum())
        })

    except Exception as e:
        print("CSV ERROR:", str(e))

# --------------------------------------------------
# 4. INSPECT TEXT AND JSON FILES
# --------------------------------------------------

for _, row in inventory_df.iterrows():

    path = Path(row["Full_Path"])

    if path.suffix.lower() not in [".txt", ".json", ".md"]:
        continue

    print("\n" + "="*100)
    print("TEXT FILE:", path.relative_to(BASE))
    print("="*100)

    try:
        content = path.read_text(
            encoding="utf-8",
            errors="replace"
        )

        print(content[:12000])

    except Exception as e:
        print("READ ERROR:", str(e))

# --------------------------------------------------
# 5. PDF PAGE COUNT
# --------------------------------------------------

pdf_reports = []

try:
    from pypdf import PdfReader

    for _, row in inventory_df.iterrows():

        path = Path(row["Full_Path"])

        if path.suffix.lower() != ".pdf":
            continue

        try:
            reader = PdfReader(str(path))

            pdf_reports.append({
                "File": str(path.relative_to(BASE)),
                "Pages": len(reader.pages),
                "Encrypted": reader.is_encrypted
            })

            print(
                "\nPDF:",
                path.relative_to(BASE),
                "| Pages:",
                len(reader.pages)
            )

        except Exception as e:
            pdf_reports.append({
                "File": str(path.relative_to(BASE)),
                "Pages": None,
                "Encrypted": None,
                "Error": str(e)
            })

except ImportError:
    print("\nInstall pypdf to inspect PDF page counts if needed.")

# --------------------------------------------------
# 6. SAVE REPORTS
# --------------------------------------------------

inventory_df.to_csv(
    OUTPUT / "Complete_File_Inventory.csv",
    index=False
)

folders_df.to_csv(
    OUTPUT / "Complete_Folder_Inventory.csv",
    index=False
)

pd.DataFrame(csv_reports).to_csv(
    OUTPUT / "CSV_Content_Summary.csv",
    index=False
)

pd.DataFrame(pdf_reports).to_csv(
    OUTPUT / "PDF_Page_Summary.csv",
    index=False
)

print("\n" + "="*100)
print("INSPECTION COMPLETED")
print("="*100)
print("Output folder:", OUTPUT)
print("Files inventoried:", len(inventory_df))
print("Folders inventoried:", len(folders_df))
print("CSV files inspected:", len(csv_reports))
print("PDF files inspected:", len(pdf_reports))


**[orig #256]** Package the two BDR Word folders (Atal, NHP) as a ZIP with an inventory  
<sub>• output not saved in the source notebook</sub>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import zipfile
import os

BASE = Path('/content/drive/MyDrive/gw_framework/data/BDR/Alwar')

FOLDERS = ['Atalbdr', 'nhpbdr 2023']

records = []

for folder_name in FOLDERS:
    folder = BASE / folder_name

    if not folder.exists():
        print("MISSING:", folder)
        continue

    for p in sorted(folder.rglob('*')):
        if p.is_file():
            records.append({
                'Folder': folder_name,
                'Relative_Path': str(p.relative_to(BASE)),
                'Filename': p.name,
                'Extension': p.suffix.lower(),
                'Size_MB': round(p.stat().st_size / (1024**2), 3)
            })

inventory = pd.DataFrame(records)

print("Total files:", len(inventory))
print("\nFiles by folder and type:")
display(
    inventory.groupby(['Folder', 'Extension'])
    .size().reset_index(name='File_Count')
)

print("\nComplete inventory:")
display(inventory)

inventory.to_csv(
    BASE / 'Alwar_BDR_Word_Folder_Inventory.csv',
    index=False
)

# Package both folders without modifying the originals
zip_path = BASE / 'Alwar_BDR_Word_Folders.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder_name in FOLDERS:
        folder = BASE / folder_name
        if folder.exists():
            for p in folder.rglob('*'):
                if p.is_file():
                    z.write(p, p.relative_to(BASE))

print("\nInventory saved:", BASE / 'Alwar_BDR_Word_Folder_Inventory.csv')
print("ZIP created:", zip_path)
print("ZIP size (MB):", round(zip_path.stat().st_size / (1024**2), 2))

**[orig #257]** Check the main extracted database workbook `Alwar_BDR_Extracted_Database.xlsx`  
<sub>• output not saved in the source notebook</sub>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd

BASE = Path(
    '/content/drive/MyDrive/gw_framework/data/BDR/Alwar/Alwar_BDR_Model_Ready'
)

print("Folder exists:", BASE.exists())

print("\nFiles:")
for p in sorted(BASE.rglob('*')):
    if p.is_file():
        print(f"{p.name:45s} {p.stat().st_size/1024:.1f} KB")

# Check the main database workbook
xlsx = BASE / 'Alwar_BDR_Extracted_Database.xlsx'

if xlsx.exists():
    xl = pd.ExcelFile(xlsx)
    print("\nExcel sheets:")
    for sheet in xl.sheet_names:
        df = pd.read_excel(xlsx, sheet_name=sheet)
        print(f"{sheet}: {len(df)} rows, {len(df.columns)} columns")
else:
    print("\nDatabase workbook not found.")


**[orig #258]** Borehole identifiers and QC flags (lithology, coordinates)  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path(
    '/content/drive/MyDrive/gw_framework/data/BDR/Alwar/Alwar_BDR_Model_Ready'
)

files = {
    'Borehole_Master': 'Borehole_Master_Combined.csv',
    'Lithology_Intervals': 'Lithology_Intervals_Combined.csv',
    'Lithology_QC': 'Lithology_Interval_QC.csv',
    'Coordinate_QC': 'NHP_Docx_vs_Excel_Coordinate_QC.csv',
    'Water_Zones': 'Water_Bearing_Zones_NHP.csv',
    'Well_Assembly': 'Well_Assembly_Combined.csv'
}

data = {}

for name, filename in files.items():
    df = pd.read_csv(BASE / filename)
    data[name] = df

    print('\n' + '='*65)
    print(name)
    print('='*65)
    print('Shape:', df.shape)
    print('Columns:', df.columns.tolist())
    print('\nFirst 5 records:')
    display(df.head())

# Check borehole identifiers
master = data['Borehole_Master']

print('\nMASTER DATABASE CHECK')
print('Total records:', len(master))

for col in master.columns:
    if any(x in col.lower() for x in ['id', 'name', 'village', 'block']):
        print('\n', col)
        print('Unique values:', master[col].nunique(dropna=True))
        print('Missing values:', master[col].isna().sum())

# Display lithology QC flags
print('\nLITHOLOGY QC SUMMARY')
qc = data['Lithology_QC']

for col in qc.columns:
    if any(x in col.lower() for x in ['flag', 'status', 'issue', 'qc']):
        print('\n', col)
        print(qc[col].value_counts(dropna=False).to_string())

# Display coordinate QC flags
print('\nCOORDINATE QC SUMMARY')
coord = data['Coordinate_QC']

for col in coord.columns:
    if any(x in col.lower() for x in ['flag', 'status', 'issue', 'qc']):
        print('\n', col)
        print(coord[col].value_counts(dropna=False).to_string())

# Save validation snapshot
summary = []

for name, df in data.items():
    summary.append({
        'Dataset': name,
        'Records': len(df),
        'Columns': len(df.columns),
        'Duplicate_Rows': int(df.duplicated().sum())
    })

summary = pd.DataFrame(summary)

display(summary)

summary.to_csv(
    BASE / 'Step3_Database_Validation_Summary.csv',
    index=False
)

print('\nValidation summary saved.')

**[orig #259]** Lithology records needing review; NHP coordinate discrepancies  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path(
    '/content/drive/MyDrive/gw_framework/data/BDR/Alwar/Alwar_BDR_Model_Ready'
)

OUT = BASE / 'Step3_QC_Review'
OUT.mkdir(exist_ok=True)

# Load source datasets
master = pd.read_csv(BASE / 'Borehole_Master_Combined.csv')
lith = pd.read_csv(BASE / 'Lithology_Intervals_Combined.csv')
qc = pd.read_csv(BASE / 'Lithology_Interval_QC.csv')
coord = pd.read_csv(BASE / 'NHP_Docx_vs_Excel_Coordinate_QC.csv')
water = pd.read_csv(BASE / 'Water_Bearing_Zones_NHP.csv')

# --------------------------------------------------
# A. LITHOLOGY RECORDS REQUIRING REVIEW
# --------------------------------------------------

review_ids = qc.loc[
    qc['Interval_QC'] == 'REVIEW', 'Borehole_ID'
]

lith_review = lith[
    lith['Borehole_ID'].isin(review_ids)
].copy()

lith_review = lith_review.merge(
    qc,
    on='Borehole_ID',
    how='left',
    suffixes=('', '_QC')
)

lith_review = lith_review.sort_values(
    ['Borehole_ID', 'Interval_From_m_bgl']
)

lith_review.to_csv(
    OUT / '01_Lithology_Intervals_Review.csv',
    index=False
)

qc.to_csv(
    OUT / '02_All_Borehole_Lithology_QC.csv',
    index=False
)

# --------------------------------------------------
# B. NHP COORDINATE DISCREPANCIES
# --------------------------------------------------

coord = coord.sort_values(
    'Coordinate_Distance_m',
    ascending=False
)

coord.to_csv(
    OUT / '03_NHP_Coordinate_Discrepancies.csv',
    index=False
)

# --------------------------------------------------
# C. BOREHOLE MASTER WITH SOURCE INFORMATION
# --------------------------------------------------

master.to_csv(
    OUT / '04_Borehole_Master_Source_Fields.csv',
    index=False
)

# --------------------------------------------------
# D. WATER-BEARING ZONE REVIEW
# --------------------------------------------------

water['Calculated_Thickness_m'] = (
    water['Interval_To_m_bgl'] -
    water['Interval_From_m_bgl']
)

water['Thickness_Difference_m'] = (
    water['Calculated_Thickness_m'] -
    water['Thickness_Reported_m']
)

water.to_csv(
    OUT / '05_Water_Bearing_Zone_Review.csv',
    index=False
)

# --------------------------------------------------
# E. SUMMARY
# --------------------------------------------------

print('LITHOLOGY QC COUNTS')
print(qc.groupby(['Dataset', 'Interval_QC']).size())

print('\nNHP COORDINATE DISTANCE SUMMARY')
print(coord['Coordinate_Distance_m'].describe())

print('\nLARGEST COORDINATE DISCREPANCIES')
display(
    coord[
        [
            'Borehole_ID',
            'Village_From_Filename',
            'Latitude_DD',
            'Longitude_DD',
            'Latitude',
            'Longitude',
            'Coordinate_Distance_m',
            'Coordinate_QC'
        ]
    ].head(15)
)

print('\nWATER ZONE RECORDS')
display(water.head(10))

print('\nReview files saved in:')
print(OUT)

**[orig #260]** Five largest coordinate discrepancies; all 21 lithology records needing review  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path(
    '/content/drive/MyDrive/gw_framework/data/BDR/Alwar/Alwar_BDR_Model_Ready'
)

OUT = BASE / 'Step4_Source_Verification'
OUT.mkdir(exist_ok=True)

# Load datasets
coord = pd.read_csv(BASE / 'NHP_Docx_vs_Excel_Coordinate_QC.csv')
qc = pd.read_csv(BASE / 'Lithology_Interval_QC.csv')
lith = pd.read_csv(BASE / 'Lithology_Intervals_Combined.csv')
master = pd.read_csv(BASE / 'Borehole_Master_Combined.csv')

# 1. Five largest coordinate discrepancies
top5 = coord.sort_values(
    'Coordinate_Distance_m', ascending=False
).head(5)

print("FIVE HIGHEST COORDINATE DISCREPANCIES")
display(top5[[
    'Borehole_ID',
    'Source_File',
    'Village_From_Filename',
    'Village_XLSX',
    'Latitude_DD',
    'Longitude_DD',
    'Latitude',
    'Longitude',
    'Coordinate_Distance_m'
]])

top5.to_csv(
    OUT / '01_Top5_Coordinate_Verification.csv',
    index=False
)

# 2. All 21 lithology records requiring review
review_ids = qc.loc[
    qc['Interval_QC'] == 'REVIEW', 'Borehole_ID'
]

review_logs = lith[
    lith['Borehole_ID'].isin(review_ids)
].sort_values(
    ['Borehole_ID', 'Interval_From_m_bgl']
)

review_logs = review_logs.merge(
    qc[['Borehole_ID', 'Overlap_Flags',
        'Gap_Flags', 'Interval_QC']],
    on='Borehole_ID',
    how='left'
)

print("\nLITHOLOGY LOGS REQUIRING REVIEW")
display(review_logs)

review_logs.to_csv(
    OUT / '02_All_21_Lithology_Review_Intervals.csv',
    index=False
)

# 3. Original source file references
source_register = master[[
    'Borehole_ID',
    'Source_Folder',
    'Source_File',
    'Latitude_DD',
    'Longitude_DD',
    'Coordinate_Source_Text',
    'Ground_RL_m',
    'Elevation_m_amsl',
    'Total_Depth_m'
]].copy()

source_register.to_csv(
    OUT / '03_Borehole_Source_Register.csv',
    index=False
)

print("\nSource verification files saved to:")
print(OUT)

## 2. NHP coordinates from the Word files; decision register

**[orig #261]** Extract DMS coordinates from the 42 NHP Word files (first version)  
<sub>• output not saved in the source notebook</sub>

In [ ]:
!pip -q install python-docx

from pathlib import Path
from docx import Document
import pandas as pd
import re
import numpy as np

BASE = Path(
    '/content/drive/MyDrive/gw_framework/data/BDR/Alwar/Alwar_BDR_Model_Ready'
)

OUT = BASE / 'Step4_Source_Verification'
OUT.mkdir(exist_ok=True)

QC = pd.read_csv(BASE / 'NHP_Docx_vs_Excel_Coordinate_QC.csv')

def dms_to_dd(deg, minute, sec, direction):
    value = float(deg) + float(minute)/60 + float(sec)/3600
    if direction.upper() in ['S', 'W']:
        value *= -1
    return value

pattern = re.compile(
    r'(\d{1,2})\s*[°º]\s*(\d{1,2})\s*[′\']\s*'
    r'([\d.]+)\s*[″"]?\s*([NSEW])',
    re.IGNORECASE
)

records = []

for _, row in QC.iterrows():

    filename = row['Source_File']
    filepath = BASE / filename

    result = {
        'Borehole_ID': row['Borehole_ID'],
        'Source_File': filename,
        'Coordinate_Text_Found': None,
        'DOCX_Latitude_DD': np.nan,
        'DOCX_Longitude_DD': np.nan,
        'Existing_DOCX_Latitude_DD': row['Latitude_DD'],
        'Existing_DOCX_Longitude_DD': row['Longitude_DD'],
        'Excel_Latitude': row['Latitude'],
        'Excel_Longitude': row['Longitude'],
        'DOCX_Extraction_Status': 'NOT_FOUND'
    }

    if filepath.exists():
        doc = Document(filepath)

        all_text = []

        for table in doc.tables:
            for table_row in table.rows:
                all_text.append(
                    ' | '.join(cell.text for cell in table_row.cells)
                )

        for paragraph in doc.paragraphs:
            all_text.append(paragraph.text)

        for text in all_text:
            if 'coordinate' in text.lower() or 'co-ordinate' in text.lower():
                matches = pattern.findall(text)

                if len(matches) >= 2:
                    lat_match = next(
                        (m for m in matches if m[3].upper() in ['N', 'S']),
                        None
                    )
                    lon_match = next(
                        (m for m in matches if m[3].upper() in ['E', 'W']),
                        None
                    )

                    if lat_match and lon_match:
                        lat = dms_to_dd(*lat_match)
                        lon = dms_to_dd(*lon_match)

                        result['Coordinate_Text_Found'] = text
                        result['DOCX_Latitude_DD'] = lat
                        result['DOCX_Longitude_DD'] = lon
                        result['DOCX_Extraction_Status'] = 'EXTRACTED'
                        break

    if result['DOCX_Extraction_Status'] == 'EXTRACTED':
        result['DOCX_Extraction_Difference_m'] = (
            111320 * np.sqrt(
                (result['DOCX_Latitude_DD'] -
                 result['Existing_DOCX_Latitude_DD'])**2 +
                (
                    (result['DOCX_Longitude_DD'] -
                     result['Existing_DOCX_Longitude_DD']) *
                    np.cos(np.radians(result['DOCX_Latitude_DD']))
                )**2
            )
        )

    records.append(result)

audit = pd.DataFrame(records)

audit.to_csv(
    OUT / '06_NHP_All42_DOCX_DMS_Coordinate_Audit.csv',
    index=False
)

print("Coordinate extraction status:")
print(audit['DOCX_Extraction_Status'].value_counts())

print("\nDOCX coordinate extraction comparison:")
display(audit[[
    'Borehole_ID',
    'DOCX_Latitude_DD',
    'DOCX_Longitude_DD',
    'Existing_DOCX_Latitude_DD',
    'Existing_DOCX_Longitude_DD',
    'Excel_Latitude',
    'Excel_Longitude',
    'DOCX_Extraction_Difference_m'
]])

print("\nSaved:", OUT)

**[orig #262]** Extract DMS coordinates from the NHP Word files (flexible pattern) → `06_NHP_All42_DOCX_DMS_Coordinate_Audit.csv`  
<sub>• output not saved in the source notebook</sub>

In [ ]:
!pip -q install python-docx

from pathlib import Path
from docx import Document
import pandas as pd
import re
import numpy as np

BASE = Path(
    '/content/drive/MyDrive/gw_framework/data/BDR/Alwar/Alwar_BDR_Model_Ready'
)

OUT = BASE / 'Step4_Source_Verification'
OUT.mkdir(exist_ok=True)

QC = pd.read_csv(BASE / 'NHP_Docx_vs_Excel_Coordinate_QC.csv')

# Flexible DMS pattern
DMS_PATTERN = re.compile(
    r'(\d{1,2})\s*[°º]\s*'
    r'(\d{1,2})\s*[\'′’]\s*'
    r'(\d{1,2}(?:\.\d+)?)\s*["″”]?\s*'
    r'([NSEW])',
    re.IGNORECASE
)

def dms_to_dd(match):
    deg, minute, sec, direction = match.groups()

    value = (
        float(deg)
        + float(minute) / 60
        + float(sec) / 3600
    )

    if direction.upper() in ['S', 'W']:
        value *= -1

    return value

records = []

for _, row in QC.iterrows():

    filename = str(row['Source_File'])
    filepath = BASE / filename

    # Search recursively if not found at the root
    if not filepath.exists():
        matches = list(BASE.rglob(filename))
        filepath = matches[0] if matches else None

    # Initialize all output fields
    rec = {
        'Borehole_ID': row['Borehole_ID'],
        'Source_File': filename,
        'Coordinate_Text': '',
        'DOCX_Latitude_DD': np.nan,
        'DOCX_Longitude_DD': np.nan,
        'Existing_DOCX_Latitude_DD': row['Latitude_DD'],
        'Existing_DOCX_Longitude_DD': row['Longitude_DD'],
        'Excel_Latitude': row['Latitude'],
        'Excel_Longitude': row['Longitude'],
        'DOCX_Extraction_Status': 'NOT_FOUND',
        'DOCX_Extraction_Difference_m': np.nan
    }

    if filepath is not None:

        doc = Document(filepath)
        text_parts = []

        # Extract paragraphs
        for p in doc.paragraphs:
            if p.text.strip():
                text_parts.append(p.text)

        # Extract all table cells
        for table in doc.tables:
            for tr in table.rows:
                for cell in tr.cells:
                    if cell.text.strip():
                        text_parts.append(cell.text)

        full_text = '\n'.join(text_parts)

        # Normalize common Unicode variations
        full_text = (
            full_text
            .replace('º', '°')
            .replace('′', "'")
            .replace('’', "'")
            .replace('″', '"')
            .replace('”', '"')
        )

        matches = DMS_PATTERN.findall(full_text)

        lat_matches = [
            m for m in matches if m[3].upper() in ['N', 'S']
        ]

        lon_matches = [
            m for m in matches if m[3].upper() in ['E', 'W']
        ]

        if lat_matches and lon_matches:

            lat_match = DMS_PATTERN.search(
                ' '.join([
                    f"{m[0]}°{m[1]}'{m[2]}\"{m[3]}"
                    for m in lat_matches
                ])
            )

            lon_match = DMS_PATTERN.search(
                ' '.join([
                    f"{m[0]}°{m[1]}'{m[2]}\"{m[3]}"
                    for m in lon_matches
                ])
            )

            if lat_match and lon_match:

                rec['DOCX_Latitude_DD'] = dms_to_dd(lat_match)
                rec['DOCX_Longitude_DD'] = dms_to_dd(lon_match)
                rec['Coordinate_Text'] = (
                    str(lat_match.group(0)) + ' ; ' +
                    str(lon_match.group(0))
                )
                rec['DOCX_Extraction_Status'] = 'EXTRACTED'

                lat1 = rec['DOCX_Latitude_DD']
                lon1 = rec['DOCX_Longitude_DD']

                lat2 = rec['Existing_DOCX_Latitude_DD']
                lon2 = rec['Existing_DOCX_Longitude_DD']

                # Approximate local distance in metres
                rec['DOCX_Extraction_Difference_m'] = (
                    111320 * np.sqrt(
                        (lat1 - lat2)**2 +
                        (
                            (lon1 - lon2) *
                            np.cos(np.radians(lat1))
                        )**2
                    )
                )

    records.append(rec)

audit = pd.DataFrame(records)

# Save
output_file = OUT / '06_NHP_All42_DOCX_DMS_Coordinate_Audit.csv'
audit.to_csv(output_file, index=False)

print('EXTRACTION STATUS')
print(audit['DOCX_Extraction_Status'].value_counts())

print('\nCOORDINATE COMPARISON')
display(audit[[
    'Borehole_ID',
    'Coordinate_Text',
    'DOCX_Latitude_DD',
    'DOCX_Longitude_DD',
    'Existing_DOCX_Latitude_DD',
    'Existing_DOCX_Longitude_DD',
    'Excel_Latitude',
    'Excel_Longitude',
    'DOCX_Extraction_Difference_m',
    'DOCX_Extraction_Status'
]])

print('\nSaved:', output_file)

**[orig #263]** NHP coordinate decision register (distance between coordinate sources)  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# PATHS
# --------------------------------------------------
BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready"
)

INPUT = (
    BASE / "Step4_Source_Verification" /
    "06_NHP_All42_DOCX_DMS_Coordinate_Audit.csv"
)

OUT = BASE / "Step4_Source_Verification"
OUT.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# LOAD AUDIT
# --------------------------------------------------
df = pd.read_csv(INPUT)

required = [
    "Borehole_ID",
    "Coordinate_Text",
    "DOCX_Latitude_DD",
    "DOCX_Longitude_DD",
    "Excel_Latitude",
    "Excel_Longitude"
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# --------------------------------------------------
# CALCULATE DISTANCE BETWEEN COORDINATE SOURCES
# --------------------------------------------------
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000  # Mean Earth radius in metres

    lat1, lon1, lat2, lon2 = np.radians(
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2)**2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(dlon / 2)**2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


df["DOCX_Excel_Distance_m"] = df.apply(
    lambda r: haversine_m(
        r["DOCX_Latitude_DD"],
        r["DOCX_Longitude_DD"],
        r["Excel_Latitude"],
        r["Excel_Longitude"]
    ),
    axis=1
)

# --------------------------------------------------
# CLASSIFY CONFLICTS
# These are review categories, not acceptance rules.
# --------------------------------------------------
def classify_distance(distance):
    if pd.isna(distance):
        return "COORDINATE_MISSING"
    elif distance <= 100:
        return "CLOSE_MATCH_WITHIN_100M"
    elif distance <= 500:
        return "MINOR_CONFLICT_100_TO_500M"
    elif distance <= 2000:
        return "MODERATE_CONFLICT_500_TO_2000M"
    else:
        return "MAJOR_CONFLICT_OVER_2000M"

df["Coordinate_Conflict_Class"] = (
    df["DOCX_Excel_Distance_m"].apply(classify_distance)
)

# --------------------------------------------------
# CREATE REVIEW / DECISION FIELDS
# --------------------------------------------------
df["DOCX_Source"] = "NHP borehole DOCX report"
df["Excel_Source"] = "NHP Excel master"

df["Verification_Status"] = "PENDING_VERIFICATION"
df["Accepted_Latitude"] = np.nan
df["Accepted_Longitude"] = np.nan
df["Accepted_Source"] = "NOT_DECIDED"

df["Evidence_Checked"] = ""
df["Reviewer_Comments"] = ""
df["Review_Date"] = ""

# --------------------------------------------------
# SORT LARGEST CONFLICTS FIRST
# --------------------------------------------------
df = df.sort_values(
    "DOCX_Excel_Distance_m",
    ascending=False
).reset_index(drop=True)

# --------------------------------------------------
# SAVE
# --------------------------------------------------
csv_path = OUT / "07_NHP_Coordinate_Decision_Register.csv"
xlsx_path = OUT / "07_NHP_Coordinate_Decision_Register.xlsx"

df.to_csv(csv_path, index=False)
df.to_excel(xlsx_path, index=False)

print("Coordinate decision register created.")
print("Total boreholes:", len(df))
print("\nConflict categories:")
print(df["Coordinate_Conflict_Class"].value_counts())

print("\nLargest coordinate conflicts:")
display(
    df[[
        "Borehole_ID",
        "DOCX_Latitude_DD",
        "DOCX_Longitude_DD",
        "Excel_Latitude",
        "Excel_Longitude",
        "DOCX_Excel_Distance_m",
        "Coordinate_Conflict_Class",
        "Verification_Status"
    ]].head(10)
)

print("\nSaved files:")
print(csv_path)
print(xlsx_path)

## 3. Lithology review against the original Word tables (Step 4)

**[orig #264]** Summary of flagged lithology intervals by borehole  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready"
)

STEP4 = BASE / "Step4_Source_Verification"

# Locate existing review file
files = list(STEP4.glob("02_All_21_Lithology_Review_Intervals.csv"))

if not files:
    raise FileNotFoundError(
        "02_All_21_Lithology_Review_Intervals.csv not found."
    )

lith = pd.read_csv(files[0])

print("Total flagged interval records:", len(lith))
print("\nColumns:")
print(lith.columns.tolist())

# Identify borehole ID column
possible_ids = ["Borehole_ID", "BoreholeID", "Well_ID"]
id_col = next((c for c in possible_ids if c in lith.columns), None)

if id_col is None:
    raise ValueError("Could not identify borehole ID column.")

# Summary by borehole
summary = (
    lith.groupby(id_col)
    .size()
    .reset_index(name="Interval_Records")
    .sort_values(id_col)
)

print("\nNumber of flagged boreholes:", len(summary))
display(summary)

# Save summary
summary.to_csv(
    STEP4 / "08_Lithology_Review_Borehole_Summary.csv",
    index=False
)

print("\nSaved:",
      STEP4 / "08_Lithology_Review_Borehole_Summary.csv")

**[orig #265]** Step 4D — extract the original Word tables for the flagged boreholes  
<sub>• output not saved in the source notebook</sub>

In [ ]:
# ============================================================
# STEP 4D: ORIGINAL DOCX TABLE EXTRACTION FOR LITHOLOGY REVIEW
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
from docx import Document

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready"
)

STEP4 = BASE / "Step4_Source_Verification"

REVIEW_FILE = (
    STEP4 / "02_All_21_Lithology_Review_Intervals.csv"
)

if not REVIEW_FILE.exists():
    raise FileNotFoundError(REVIEW_FILE)

# ------------------------------------------------------------
# 2. LOAD FLAGGED INTERVALS
# ------------------------------------------------------------

review = pd.read_csv(REVIEW_FILE)

required = [
    "Borehole_ID",
    "Source_File",
    "Interval_From_m_bgl",
    "Interval_To_m_bgl",
    "Lithology_Description_Raw",
    "Overlap_Flags",
    "Gap_Flags"
]

missing = [c for c in required if c not in review.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

print("Flagged interval records:", len(review))
print("Flagged boreholes:", review["Borehole_ID"].nunique())

# ------------------------------------------------------------
# 3. INDEX ALL ORIGINAL DOCX FILES
# ------------------------------------------------------------

docx_files = list(BASE.rglob("*.docx"))

print("DOCX files found:", len(docx_files))

# Index by filename (case-insensitive)
docx_index = {}

for p in docx_files:
    docx_index.setdefault(p.name.lower(), []).append(p)

# ------------------------------------------------------------
# 4. EXTRACT ORIGINAL TABLES
# ------------------------------------------------------------

source_rows = []
document_register = []

unique_sources = (
    review[["Borehole_ID", "Source_File"]]
    .drop_duplicates()
)

for _, record in unique_sources.iterrows():

    borehole_id = str(record["Borehole_ID"])
    source_file = str(record["Source_File"]).strip()

    filename = Path(source_file).name

    matches = docx_index.get(filename.lower(), [])

    if len(matches) == 0:
        document_register.append({
            "Borehole_ID": borehole_id,
            "Source_File": source_file,
            "Matched_Path": "",
            "Match_Status": "NOT_FOUND",
            "Number_of_Tables": 0,
            "Number_of_Table_Rows": 0
        })
        continue

    if len(matches) > 1:
        # Do not arbitrarily select one duplicate filename.
        document_register.append({
            "Borehole_ID": borehole_id,
            "Source_File": source_file,
            "Matched_Path": "; ".join(str(x) for x in matches),
            "Match_Status": "DUPLICATE_FILENAME_REVIEW",
            "Number_of_Tables": 0,
            "Number_of_Table_Rows": 0
        })
        continue

    doc_path = matches[0]

    try:
        doc = Document(str(doc_path))

        table_count = len(doc.tables)
        row_count = 0

        # Extract every row of every table.
        for table_no, table in enumerate(doc.tables, start=1):

            for row_no, row in enumerate(table.rows, start=1):

                cells = [
                    re.sub(r"\s+", " ", cell.text).strip()
                    for cell in row.cells
                ]

                row_count += 1

                source_rows.append({
                    "Borehole_ID": borehole_id,
                    "Source_File": filename,
                    "Table_Number": table_no,
                    "Row_Number": row_no,
                    "Number_of_Cells": len(cells),
                    "Original_Table_Row": " | ".join(cells),
                    "Cell_Contents": " || ".join(
                        f"Cell_{i+1}: {value}"
                        for i, value in enumerate(cells)
                    ),
                    "Document_Path": str(doc_path)
                })

        document_register.append({
            "Borehole_ID": borehole_id,
            "Source_File": filename,
            "Matched_Path": str(doc_path),
            "Match_Status": "EXTRACTED",
            "Number_of_Tables": table_count,
            "Number_of_Table_Rows": row_count
        })

    except Exception as e:

        document_register.append({
            "Borehole_ID": borehole_id,
            "Source_File": filename,
            "Matched_Path": str(doc_path),
            "Match_Status": "EXTRACTION_ERROR: " + str(e),
            "Number_of_Tables": 0,
            "Number_of_Table_Rows": 0
        })

# ------------------------------------------------------------
# 5. CREATE DATAFRAMES
# ------------------------------------------------------------

source_tables = pd.DataFrame(source_rows)
doc_register = pd.DataFrame(document_register)

# ------------------------------------------------------------
# 6. ADD A SORTED INTERVAL REVIEW SHEET
# ------------------------------------------------------------

review_sorted = review.copy()

review_sorted = review_sorted.sort_values(
    ["Dataset", "Borehole_ID",
     "Interval_From_m_bgl", "Interval_To_m_bgl"],
    na_position="last"
)

# ------------------------------------------------------------
# 7. SAVE OUTPUTS
# ------------------------------------------------------------

csv_tables = STEP4 / "09_Original_DOCX_Table_Extraction.csv"
csv_register = STEP4 / "09_Original_DOCX_Document_Register.csv"

xlsx_output = STEP4 / "09_Lithology_Source_Comparison_Workbook.xlsx"

source_tables.to_csv(csv_tables, index=False)
doc_register.to_csv(csv_register, index=False)

with pd.ExcelWriter(xlsx_output, engine="openpyxl") as writer:

    review_sorted.to_excel(
        writer,
        sheet_name="Flagged_Intervals",
        index=False
    )

    source_tables.to_excel(
        writer,
        sheet_name="Original_DOCX_Tables",
        index=False
    )

    doc_register.to_excel(
        writer,
        sheet_name="Document_Register",
        index=False
    )

# ------------------------------------------------------------
# 8. REPORT
# ------------------------------------------------------------

print("\nDOCUMENT MATCH STATUS")
print(doc_register["Match_Status"].value_counts(dropna=False))

print("\nDOCUMENT REGISTER")
display(doc_register)

print("\nEXTRACTED ORIGINAL TABLE ROWS:", len(source_tables))

print("\nOUTPUT FILES")
print(csv_tables)
print(csv_register)
print(xlsx_output)

**[orig #266]** Identify likely lithology/drilling table rows  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import pandas as pd
import re
from pathlib import Path

# --------------------------------------------------
# PATHS
# --------------------------------------------------
STEP4 = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready/Step4_Source_Verification"
)

INPUT = STEP4 / "09_Original_DOCX_Table_Extraction.csv"

# --------------------------------------------------
# LOAD ORIGINAL TABLE ROWS
# --------------------------------------------------
source_tables = pd.read_csv(INPUT).fillna("")

print("Original extracted rows:", len(source_tables))
print("Boreholes:", source_tables["Borehole_ID"].nunique())

# --------------------------------------------------
# IDENTIFY LIKELY LITHOLOGY / DRILLING TABLE ROWS
# --------------------------------------------------
keywords = [
    "litholog", "lithology", "depth", "bgl",
    "soil", "clay", "sand", "gravel", "silt",
    "weathered", "fractured", "shale", "quartzite",
    "gneiss", "granite", "schist", "rock",
    "boulder", "formation", "from", "to"
]

pattern = re.compile("|".join(re.escape(x) for x in keywords), re.I)

source_tables["Keyword_Match"] = source_tables[
    "Original_Table_Row"
].apply(lambda x: bool(pattern.search(str(x))))

candidates = source_tables[
    source_tables["Keyword_Match"]
].copy()

# --------------------------------------------------
# TABLE-LEVEL SUMMARY
# --------------------------------------------------
table_summary = (
    candidates.groupby(
        ["Borehole_ID", "Source_File", "Table_Number"]
    )
    .agg(
        Candidate_Rows=("Row_Number", "count"),
        First_Row=("Row_Number", "min"),
        Last_Row=("Row_Number", "max")
    )
    .reset_index()
    .sort_values(["Borehole_ID", "Table_Number"])
)

print("\nCandidate rows:", len(candidates))
print("Boreholes represented:", candidates["Borehole_ID"].nunique())

print("\nCandidate table summary:")
display(table_summary)

# --------------------------------------------------
# SAVE CANDIDATES
# --------------------------------------------------
candidate_file = STEP4 / "10_Lithology_Table_Candidate_Rows.csv"
summary_file = STEP4 / "10_Lithology_Table_Candidate_Summary.csv"

candidates.to_csv(candidate_file, index=False)
table_summary.to_csv(summary_file, index=False)

print("\nSaved:")
print(candidate_file)
print(summary_file)

# --------------------------------------------------
# DISPLAY TWO EXAMPLE BOREHOLES
# --------------------------------------------------
for borehole in [
    "ATAL/0522/ALW/RAJ/03",
    "NHP-146"
]:
    print(f"\n{'='*70}\n{borehole}\n{'='*70}")

    display(
        candidates[
            candidates["Borehole_ID"] == borehole
        ][[
            "Table_Number",
            "Row_Number",
            "Original_Table_Row"
        ]].to_string(index=False)
    )

**[orig #267]** Show the original tables for two boreholes (ATAL/0522/ALW/RAJ/03, NHP-146)  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# PATHS
# --------------------------------------------------
STEP4 = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready/Step4_Source_Verification"
)

SOURCE_FILE = STEP4 / "09_Original_DOCX_Table_Extraction.csv"
REVIEW_FILE = STEP4 / "02_All_21_Lithology_Review_Intervals.csv"

# --------------------------------------------------
# LOAD DATA
# --------------------------------------------------
source = pd.read_csv(SOURCE_FILE).fillna("")
review = pd.read_csv(REVIEW_FILE).fillna("")

# --------------------------------------------------
# ATAL BOREHOLE: ORIGINAL TABLE 2
# --------------------------------------------------
print("=" * 80)
print("ATAL/0522/ALW/RAJ/03 — ORIGINAL TABLE 2")
print("=" * 80)

atal = source[
    (source["Borehole_ID"] == "ATAL/0522/ALW/RAJ/03")
    & (source["Table_Number"] == 2)
].sort_values("Row_Number")

for _, r in atal.iterrows():
    print(
        "Row", r["Row_Number"], ":",
        r["Original_Table_Row"]
    )

print("\nEXTRACTED ATAL INTERVALS")

atal_review = review[
    review["Borehole_ID"] == "ATAL/0522/ALW/RAJ/03"
]

display(
    atal_review[
        [
            "Interval_From_m_bgl",
            "Interval_To_m_bgl",
            "Thickness_Reported_m",
            "Lithology_Description_Raw",
            "Overlap_Flags",
            "Gap_Flags",
            "Interval_QC"
        ]
    ]
)

# --------------------------------------------------
# NHP BOREHOLE: ALL ORIGINAL TABLES
# --------------------------------------------------
print("\n" + "=" * 80)
print("NHP-146 — ALL ORIGINAL TABLES")
print("=" * 80)

nhp = source[
    source["Borehole_ID"] == "NHP-146"
].sort_values(
    ["Table_Number", "Row_Number"]
)

for table_no, group in nhp.groupby("Table_Number"):

    print(f"\n--- TABLE {table_no} ---")

    for _, r in group.iterrows():
        print(
            "Row", r["Row_Number"], ":",
            r["Original_Table_Row"]
        )

# --------------------------------------------------
# NHP EXTRACTED INTERVALS
# --------------------------------------------------
print("\nEXTRACTED NHP-146 INTERVALS")

nhp_review = review[
    review["Borehole_ID"] == "NHP-146"
]

display(
    nhp_review[
        [
            "Interval_From_m_bgl",
            "Interval_To_m_bgl",
            "Thickness_Reported_m",
            "Lithology_Description_Raw",
            "Overlap_Flags",
            "Gap_Flags",
            "Interval_QC"
        ]
    ]
)

**[orig #270]** Check that the key lithology files exist  
<sub>• output not saved in the source notebook</sub>

In [ ]:
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/Alwar_BDR_Model_Ready"
)

search_files = [
    "11_Parsed_Original_Lithology_Intervals.csv",
    "Alwar_Accepted_Lithology_Intervals.csv",
    "Alwar_Lithology_Classified_Intervals_Final.csv",
    "Step6C_SRTM30m_Borehole_Elevations.csv",
    "Borehole_Master_Combined.csv"
]

for filename in search_files:
    print("\n" + "="*65)
    print("FILE:", filename)

    matches = list(BASE.rglob(filename))

    if matches:
        for path in matches:
            print("FOUND:", path)
    else:
        print("NOT FOUND")

**[orig #271]** Check the output folder (needs `BASE` from the cell above)  
<sub>• output not saved in the source notebook</sub>

In [ ]:
from pathlib import Path

OUTPUT_DIR = (
    BASE / "MODFLOW_Model_Input" /
    "Step6_Elevation_Validation"
)

print("Output folder exists:", OUTPUT_DIR.exists())

if OUTPUT_DIR.exists():
    for f in sorted(OUTPUT_DIR.iterdir()):
        print(f.name)

**[orig #272]** **MAIN** Step 4H — source-preserving lithology audit (no automatic boundary correction)  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ======================================================
# STEP 4H — SOURCE-PRESERVING LITHOLOGY AUDIT
# No automatic boundary correction
# ======================================================

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready"
)

STEP4 = BASE / "Step4_Source_Verification"

INPUT = STEP4 / "11_Parsed_Original_Lithology_Intervals.csv"

OUTPUT = STEP4 / "12_Lithology_Source_Preserving_Audit.csv"
LOG = STEP4 / "12_Lithology_Boundary_Audit_Log.csv"
REVIEW = STEP4 / "12_Lithology_Manual_Review.csv"

TOLERANCE = 1.00

# ------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------

df = pd.read_csv(INPUT)

df = df.sort_values(
    ["Borehole_ID", "Source_From_m", "Source_To_m"]
).reset_index(drop=True)

required = [
    "Borehole_ID",
    "Source_From_m",
    "Source_To_m",
    "Source_Thickness_m"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

# Preserve all original measurements
df["Audit_From_m"] = df["Source_From_m"]
df["Audit_To_m"] = df["Source_To_m"]

df["Boundary_Status"] = "FIRST_INTERVAL"
df["Boundary_Difference_m"] = np.nan
df["Boundary_Review_Reason"] = ""

df["Thickness_Difference_m"] = (
    df["Source_Thickness_m"] -
    (df["Source_To_m"] - df["Source_From_m"])
)

df["Thickness_Status"] = np.where(
    df["Source_Thickness_m"].isna(),
    "REPORTED_THICKNESS_MISSING",
    np.where(
        np.abs(df["Thickness_Difference_m"]) <= 0.01,
        "THICKNESS_MATCH",
        "THICKNESS_MISMATCH"
    )
)

# ------------------------------------------------------
# 2. AUDIT EACH BOREHOLE
# ------------------------------------------------------

boundary_log = []

for borehole, group in df.groupby("Borehole_ID", sort=False):

    indices = list(group.index)

    previous_bottom = None
    previous_valid = False

    for idx in indices:

        top = pd.to_numeric(
            pd.Series([df.loc[idx, "Source_From_m"]]),
            errors="coerce"
        ).iloc[0]

        bottom = pd.to_numeric(
            pd.Series([df.loc[idx, "Source_To_m"]]),
            errors="coerce"
        ).iloc[0]

        if pd.isna(top) or pd.isna(bottom) or bottom <= top:

            df.loc[idx, "Boundary_Status"] = "INVALID_INTERVAL"
            df.loc[idx, "Boundary_Review_Reason"] = (
                "Missing or invalid depth values"
            )

            previous_valid = False
            previous_bottom = None
            continue

        if previous_bottom is None:

            df.loc[idx, "Boundary_Status"] = "FIRST_INTERVAL"

        else:

            difference = top - previous_bottom

            df.loc[idx, "Boundary_Difference_m"] = difference

            if abs(difference) <= 0.01:

                status = "CONTIGUOUS"
                reason = "Intervals meet"

            elif difference > 0:

                status = "GAP"
                reason = "Gap between consecutive intervals"

            else:

                status = "OVERLAP"
                reason = "Overlap between consecutive intervals"

            df.loc[idx, "Boundary_Status"] = status
            df.loc[idx, "Boundary_Review_Reason"] = reason

            boundary_log.append({
                "Borehole_ID": borehole,
                "Previous_Interval_Bottom_m": previous_bottom,
                "Current_Interval_Top_m": top,
                "Difference_m": difference,
                "Boundary_Status": status,
                "Review_Reason": reason
            })

        # Advance using the original source bottom.
        # No corrected depth is generated.
        previous_bottom = bottom
        previous_valid = True

# ------------------------------------------------------
# 3. MANUAL REVIEW
# ------------------------------------------------------

review = df[
    (df["Boundary_Status"].isin(["GAP", "OVERLAP", "INVALID_INTERVAL"])) |
    (df["Thickness_Status"] == "THICKNESS_MISMATCH")
].copy()

boundary_log = pd.DataFrame(boundary_log)

# ------------------------------------------------------
# 4. SAVE
# ------------------------------------------------------

df.to_csv(OUTPUT, index=False)
boundary_log.to_csv(LOG, index=False)
review.to_csv(REVIEW, index=False)

# ------------------------------------------------------
# 5. SUMMARY
# ------------------------------------------------------

print("TOTAL ORIGINAL INTERVALS:", len(df))

print("\nBoundary status:")
print(df["Boundary_Status"].value_counts())

print("\nThickness status:")
print(df["Thickness_Status"].value_counts())

print("\nBoreholes:")
print(df["Borehole_ID"].nunique())

print("\nReview records:", len(review))

print("\nSaved:")
print(OUTPUT)
print(LOG)
print(REVIEW)

display(review[[
    "Borehole_ID",
    "Source_From_m",
    "Source_To_m",
    "Source_Thickness_m",
    "Boundary_Status",
    "Boundary_Review_Reason",
    "Thickness_Status",
    "Thickness_Difference_m"
]])

**[orig #273]** Compare interval fields between the audit and the accepted dataset  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready"
)

STEP4 = BASE / "Step4_Source_Verification"
MODEL = BASE / "MODFLOW_Model_Input"

audit_file = STEP4 / "12_Lithology_Source_Preserving_Audit.csv"
accepted_file = MODEL / "Alwar_Accepted_Lithology_Intervals.csv"

audit = pd.read_csv(audit_file)
accepted = pd.read_csv(accepted_file)

print("Audit intervals:", len(audit))
print("Accepted intervals:", len(accepted))

print("\nColumns in accepted dataset:")
print(accepted.columns.tolist())

print("\nColumns in source audit:")
print(audit.columns.tolist())

# Check the interval fields before comparison
for c in ["Borehole_ID", "Source_From_m", "Source_To_m"]:
    if c not in audit.columns:
        raise ValueError(f"Missing {c} in audit dataset")

# Accepted dataset may use corrected interval column names
print("\nAccepted depth columns:")
print([
    c for c in accepted.columns
    if "From" in c or "To" in c
])

**[orig #274]** Step 6E — verify accepted lithology intervals against the source audit  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready"
)

STEP4 = BASE / "Step4_Source_Verification"
MODEL = BASE / "MODFLOW_Model_Input"

audit = pd.read_csv(
    STEP4 / "12_Lithology_Source_Preserving_Audit.csv"
)

accepted = pd.read_csv(
    MODEL / "Alwar_Accepted_Lithology_Intervals.csv"
)

# --------------------------------------------------
# 1. CHECK RECORD IDENTIFIERS
# --------------------------------------------------

KEY = ["Borehole_ID", "Source_Row"]

for name, data in [("Audit", audit), ("Accepted", accepted)]:
    if data[KEY].isna().any().any():
        raise ValueError(f"{name}: missing record identifiers")

    if data.duplicated(KEY).any():
        raise ValueError(f"{name}: duplicate record identifiers")

# --------------------------------------------------
# 2. MATCH ACCEPTED RECORDS TO SOURCE AUDIT
# --------------------------------------------------

comparison = accepted.merge(
    audit,
    on=KEY,
    how="left",
    suffixes=("_Accepted", "_Audit"),
    indicator=True,
    validate="one_to_one"
)

print("Matched accepted records:", (comparison["_merge"] == "both").sum())
print("Unmatched accepted records:", (comparison["_merge"] != "both").sum())

# --------------------------------------------------
# 3. COMPARE ORIGINAL SOURCE VALUES
# --------------------------------------------------

fields = [
    "Source_From_m",
    "Source_To_m",
    "Source_Thickness_m",
    "Source_Lithology"
]

for field in fields:
    a = comparison[f"{field}_Accepted"]
    b = comparison[f"{field}_Audit"]

    if field == "Source_Lithology":
        comparison[f"{field}_Match"] = (
            a.fillna("").astype(str).str.strip()
            == b.fillna("").astype(str).str.strip()
        )
    else:
        comparison[f"{field}_Match"] = (
            np.isclose(
                pd.to_numeric(a, errors="coerce"),
                pd.to_numeric(b, errors="coerce"),
                atol=0.001,
                equal_nan=True
            )
        )

match_columns = [f"{f}_Match" for f in fields]

comparison["Source_Record_Match"] = (
    comparison[match_columns].all(axis=1)
)

# --------------------------------------------------
# 4. CHECK CORRECTED DEPTHS
# --------------------------------------------------

comparison["Corrected_From_Equals_Source"] = np.isclose(
    pd.to_numeric(comparison["Corrected_From_m"], errors="coerce"),
    pd.to_numeric(comparison["Source_From_m_Accepted"], errors="coerce"),
    atol=0.001,
    equal_nan=True
)

comparison["Corrected_To_Equals_Source"] = np.isclose(
    pd.to_numeric(comparison["Corrected_To_m"], errors="coerce"),
    pd.to_numeric(comparison["Source_To_m_Accepted"], errors="coerce"),
    atol=0.001,
    equal_nan=True
)

comparison["Depths_Unchanged"] = (
    comparison["Corrected_From_Equals_Source"]
    & comparison["Corrected_To_Equals_Source"]
)

# --------------------------------------------------
# 5. FINAL STATUS
# --------------------------------------------------

comparison["Verification_Status"] = np.select(
    [
        comparison["_merge"] != "both",
        ~comparison["Source_Record_Match"],
        ~comparison["Depths_Unchanged"]
    ],
    [
        "SOURCE_RECORD_NOT_FOUND",
        "SOURCE_VALUES_MISMATCH",
        "CORRECTED_DEPTHS_DIFFER"
    ],
    default="VERIFIED_UNCHANGED"
)

# --------------------------------------------------
# 6. SAVE
# --------------------------------------------------

output = MODEL / "Step6E_Accepted_Lithology_Source_Verification.csv"

comparison.to_csv(output, index=False)

print("\nVerification summary:")
print(comparison["Verification_Status"].value_counts())

print("\nSource record matches:",
      comparison["Source_Record_Match"].sum())

print("Unchanged depths:",
      comparison["Depths_Unchanged"].sum())

print("\nSaved:", output)

display(
    comparison[
        comparison["Verification_Status"] != "VERIFIED_UNCHANGED"
    ][[
        "Borehole_ID",
        "Source_Row",
        "Verification_Status",
        "Source_From_m_Accepted",
        "Source_To_m_Accepted",
        "Corrected_From_m",
        "Corrected_To_m"
    ]]
)

## 4. Borehole lithology elevation database (Step 6)

**[orig #276]** **MAIN** Step 6F — borehole lithology elevation database  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================================
# STEP 6F: BOREHOLE LITHOLOGY ELEVATION DATABASE
# ==========================================================

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready"
)

MODEL = BASE / "MODFLOW_Model_Input"

ELEV_DIR = MODEL / "Step6_Elevation_Validation"
ELEV_DIR.mkdir(parents=True, exist_ok=True)

# Input files
LITH_FILE = MODEL / "Alwar_Accepted_Lithology_Intervals.csv"

MASTER_FILE = BASE / "Borehole_Master_Combined.csv"

DEM_FILE = (
    ELEV_DIR / "Step6C_SRTM30m_Borehole_Elevations.csv"
)

VERIFY_FILE = (
    MODEL / "Step6E_Accepted_Lithology_Source_Verification.csv"
)

# Output files
OUTPUT = (
    ELEV_DIR / "Step6F_Borehole_Lithology_Elevation_Database.csv"
)

SUMMARY = (
    ELEV_DIR / "Step6F_Borehole_Elevation_Summary.csv"
)

# ----------------------------------------------------------
# 1. LOAD DATA
# ----------------------------------------------------------

lith = pd.read_csv(LITH_FILE)
master = pd.read_csv(MASTER_FILE)
dem = pd.read_csv(DEM_FILE)
verification = pd.read_csv(VERIFY_FILE)

print("Accepted lithology intervals:", len(lith))
print("Master boreholes:", master["Borehole_ID"].nunique())
print("DEM boreholes:", dem["Borehole_ID"].nunique())

# ----------------------------------------------------------
# 2. VERIFY ACCEPTED LITHOLOGY
# ----------------------------------------------------------

if len(lith) != 147:
    raise ValueError("Expected 147 accepted lithology intervals.")

if len(verification) != 147:
    raise ValueError("Expected 147 source-verification records.")

if not (
    verification["Verification_Status"] == "VERIFIED_UNCHANGED"
).all():
    raise ValueError("Lithology source verification did not pass.")

# Use original source depths, not corrected depths
lith["Depth_From_m_bgl"] = pd.to_numeric(
    lith["Source_From_m"], errors="coerce"
)

lith["Depth_To_m_bgl"] = pd.to_numeric(
    lith["Source_To_m"], errors="coerce"
)

lith["Interval_Thickness_m"] = (
    lith["Depth_To_m_bgl"] - lith["Depth_From_m_bgl"]
)

if (
    lith["Depth_From_m_bgl"].isna().any()
    or lith["Depth_To_m_bgl"].isna().any()
    or (lith["Interval_Thickness_m"] <= 0).any()
):
    raise ValueError("Invalid lithology depth interval detected.")

# ----------------------------------------------------------
# 3. PREPARE MASTER ELEVATION DATA
# ----------------------------------------------------------

master["Ground_RL_m"] = pd.to_numeric(
    master["Ground_RL_m"], errors="coerce"
)

# Keep only necessary fields
master_elev = master[[
    "Borehole_ID",
    "Ground_RL_m"
]].copy()

if master_elev["Borehole_ID"].duplicated().any():
    raise ValueError("Duplicate borehole IDs in master.")

# ----------------------------------------------------------
# 4. PREPARE DEM DATA
# ----------------------------------------------------------

dem["SRTM30m_Elevation_m"] = pd.to_numeric(
    dem["SRTM30m_Elevation_m"], errors="coerce"
)

dem_fields = [
    "Borehole_ID",
    "SRTM30m_Elevation_m",
    "Coordinate_Source",
    "DEM_Status"
]

dem_subset = dem[dem_fields].copy()

if dem_subset["Borehole_ID"].duplicated().any():
    raise ValueError("Duplicate borehole IDs in DEM database.")

# ----------------------------------------------------------
# 5. MERGE ELEVATION DATA
# ----------------------------------------------------------

df = lith.merge(
    master_elev,
    on="Borehole_ID",
    how="left",
    validate="many_to_one"
)

df = df.merge(
    dem_subset,
    on="Borehole_ID",
    how="left",
    validate="many_to_one"
)

# ----------------------------------------------------------
# 6. ASSIGN ELEVATION PROVENANCE
# ----------------------------------------------------------

df["Elevation_Source_Selected"] = np.select(
    [
        df["Ground_RL_m"].notna(),
        df["SRTM30m_Elevation_m"].notna()
    ],
    [
        "REPORTED_GROUND_RL",
        "SRTM30M_DEM_ESTIMATE"
    ],
    default="ELEVATION_MISSING"
)

df["Elevation_Confidence_Status"] = np.select(
    [
        df["Ground_RL_m"].notna(),
        df["SRTM30m_Elevation_m"].notna()
    ],
    [
        "REPORTED_RL_DATUM_UNVERIFIED",
        "DEM_ESTIMATE_NOT_SURVEYED"
    ],
    default="NO_USABLE_ELEVATION"
)

# Selected candidate:
# ATAL: reported Ground_RL_m
# NHP: SRTM estimate, clearly flagged

df["Ground_Elevation_Candidate_m"] = np.where(
    df["Ground_RL_m"].notna(),
    df["Ground_RL_m"],
    df["SRTM30m_Elevation_m"]
)

# ----------------------------------------------------------
# 7. CALCULATE ELEVATIONS USING SELECTED CANDIDATE
# ----------------------------------------------------------

df["Lithology_Top_Elevation_Candidate_m"] = (
    df["Ground_Elevation_Candidate_m"]
    - df["Depth_From_m_bgl"]
)

df["Lithology_Bottom_Elevation_Candidate_m"] = (
    df["Ground_Elevation_Candidate_m"]
    - df["Depth_To_m_bgl"]
)

# ----------------------------------------------------------
# 8. CALCULATE PARALLEL SRTM ELEVATION SCENARIO
# ----------------------------------------------------------

df["Lithology_Top_Elevation_SRTM_m"] = (
    df["SRTM30m_Elevation_m"]
    - df["Depth_From_m_bgl"]
)

df["Lithology_Bottom_Elevation_SRTM_m"] = (
    df["SRTM30m_Elevation_m"]
    - df["Depth_To_m_bgl"]
)

# ----------------------------------------------------------
# 9. VALIDATION FLAGS
# ----------------------------------------------------------

df["Elevation_Calculation_Status"] = np.where(
    df["Ground_Elevation_Candidate_m"].notna(),
    "CALCULATED_PROVISIONAL",
    "ELEVATION_UNAVAILABLE"
)

df["Suitable_For_Final_Model_Elevation"] = "NO_PENDING_VALIDATION"

# ----------------------------------------------------------
# 10. CHECK COUNTS
# ----------------------------------------------------------

print("\n========== STEP 6F SUMMARY ==========")

print("Total intervals:", len(df))

print(
    "Boreholes:",
    df["Borehole_ID"].nunique()
)

print("\nElevation source:")
print(df.groupby("Borehole_ID")[
    "Elevation_Source_Selected"
].first().value_counts())

print("\nElevation confidence:")
print(df.groupby("Borehole_ID")[
    "Elevation_Confidence_Status"
].first().value_counts())

# Verify that all accepted boreholes have a candidate elevation
missing = df[
    df["Ground_Elevation_Candidate_m"].isna()
]

if not missing.empty:
    print("\nMissing elevation records:")
    display(missing[["Borehole_ID"]].drop_duplicates())
    raise ValueError("Some lithology boreholes lack elevations.")

# ----------------------------------------------------------
# 11. SAVE FULL INTERVAL DATABASE
# ----------------------------------------------------------

df.to_csv(OUTPUT, index=False)

# ----------------------------------------------------------
# 12. CREATE BOREHOLE SUMMARY
# ----------------------------------------------------------

summary = df.groupby("Borehole_ID").agg(
    Dataset=("Dataset", "first"),
    Number_of_Lithology_Intervals=("Source_From_m", "count"),
    Ground_RL_m=("Ground_RL_m", "first"),
    SRTM30m_Elevation_m=("SRTM30m_Elevation_m", "first"),
    Ground_Elevation_Candidate_m=(
        "Ground_Elevation_Candidate_m", "first"
    ),
    Elevation_Source=("Elevation_Source_Selected", "first"),
    Elevation_Status=("Elevation_Confidence_Status", "first"),
    Shallowest_Depth_m_bgl=("Depth_From_m_bgl", "min"),
    Maximum_Lithology_Depth_m_bgl=("Depth_To_m_bgl", "max"),
    Shallowest_Lithology_Elevation_m=(
        "Lithology_Top_Elevation_Candidate_m", "max"
    ),
    Deepest_Lithology_Elevation_m=(
        "Lithology_Bottom_Elevation_Candidate_m", "min"
    )
).reset_index()

summary.to_csv(SUMMARY, index=False)

# ----------------------------------------------------------
# 13. DISPLAY RESULTS
# ----------------------------------------------------------

print("\nSaved interval database:")
print(OUTPUT)

print("\nSaved borehole summary:")
print(SUMMARY)

print("\nBorehole elevation summary:")
display(summary)

print("\nFirst 15 lithology elevation records:")
display(df[[
    "Borehole_ID",
    "Source_Lithology",
    "Depth_From_m_bgl",
    "Depth_To_m_bgl",
    "Ground_Elevation_Candidate_m",
    "Lithology_Top_Elevation_Candidate_m",
    "Lithology_Bottom_Elevation_Candidate_m",
    "Elevation_Confidence_Status"
]].head(15))

print("\nSTEP 6F COMPLETED")

**[orig #277]** Compare classified and accepted interval tables  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready"
)

MODEL = BASE / "MODFLOW_Model_Input"

CLASS_FILE = (
    MODEL / "Alwar_Lithology_Classified_Intervals_Final.csv"
)

ACCEPTED_FILE = (
    MODEL / "Alwar_Accepted_Lithology_Intervals.csv"
)

classified = pd.read_csv(CLASS_FILE)
accepted = pd.read_csv(ACCEPTED_FILE)

print("Classified intervals:", len(classified))
print("Accepted intervals:", len(accepted))

print("\nClassified dataset columns:")
print(classified.columns.tolist())

print("\nAccepted dataset columns:")
print(accepted.columns.tolist())

print("\nClassified dataset sample:")
display(classified.head(5))

print("\nClassification-related columns:")
display([
    c for c in classified.columns
    if any(x in c.lower() for x in
           ["class", "confidence", "review", "lithology"])
])